# 1. Classifier training

This file trains and evaluates the classifiers that predict whether an OCRonos-corrected pivot sentence is
morphosyntactically analyzable (class 1) or not (class 0), using two ground-truth-free metrics
(OOV rate and perplexity).

This code reproduces Tables 1-4 of the paper and saves the final model
(logistic regression, all four features) to `outputs/best_model.pkl`, which is used by
`2_document_level_analysis.ipynb`.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from ocr_metrics import FEATURE_SETS, add_features

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TARGET = "annot_postCorrection"  # 1 = analyzable, 0 = not analyzable (after OCRonos correction)

print("scikit-learn", sklearn.__version__)  # results in the paper: 1.9.0

W1117 17:38:53.625000 42508 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


scikit-learn 1.9.0


## Annotated data

1,134 examples: a first random selection of 1,000 examples stratified by period (`annot == 1`)
and a second selection of 134 additional examples (`annot == 2`) used to increase the share of class 0.

In [2]:
df = pd.read_csv(DATA_DIR / "annotations_withPeriod.csv", sep="\t", dtype={"doc_id": str})
df = df.dropna(subset=["pivot_sent_corrected"]).reset_index(drop=True)
print(df.shape)
print("Distinct documents:", df["doc_id"].nunique())

# Table 1: annotated examples by period and selection, period is defined by "Date_correction"
pd.crosstab(df["period"], df["annot"], margins=True)

(1134, 13)
Distinct documents: 1062


annot,1,2,All
period,,,
1650-1779,300,76,376
1780-1840,200,27,227
1841-1913,225,18,243
1914-2020,275,13,288
All,1000,134,1134


In [3]:
# Table 2: quality of the original vs. post-corrected transcription
pd.crosstab(df["annot_preCorrection"], df["annot_postCorrection"], margins=True)

annot_postCorrection,0,1,All
annot_preCorrection,,,
0,255,201,456
1,26,652,678
All,281,853,1134


## Features

OOV rate and perplexity are computed on the original (`*_avant`) and post-corrected (`*_apres`)
pivot sentences; gains are original − corrected (on log-perplexity for perplexity).
This step downloads the GPT-2 model and takes a few minutes on CPU.

In [4]:
df = add_features(df)

# Rounding kept as in the original run (affects the last decimals of the scores).
rounded = ["Score_OOV_avant", "Score_OOV_apres", "Gain_Score_OOV",
           "Score_Perplexite_avant", "Score_Perplexite_apres",
           "Score_Perplexite_avant_log", "Gain_Score_Perplexite_log"]
df[rounded] = df[rounded].round(4)

df.to_csv(OUTPUT_DIR / "annotations_with_features.csv", index=False)

## Model comparison (Table 3)

Same stratified 80/20 split for every configuration; features are standardized on the training set;
both algorithms use balanced class weights.

In [5]:
y = df[TARGET]
idx_train, idx_test = train_test_split(
    df.index, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", y[idx_train].value_counts().to_dict(), "| Test:", y[idx_test].value_counts().to_dict())

MODELS = {
    "Logistic Regression": lambda: LogisticRegression(
        class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": lambda: RandomForestClassifier(
        n_estimators=300, max_depth=5, class_weight="balanced", random_state=RANDOM_STATE
    ),
}


def fit_and_predict(model_name, features):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(df.loc[idx_train, features])
    X_test = scaler.transform(df.loc[idx_test, features])
    model = MODELS[model_name]()
    model.fit(X_train, y[idx_train])
    return scaler, model, model.predict(X_test)


rows = []
for model_name in MODELS:
    for set_name, features in FEATURE_SETS.items():
        _, _, pred = fit_and_predict(model_name, features)
        y_test = y[idx_test]
        f1_0, f1_1 = f1_score(y_test, pred, average=None, labels=[0, 1])
        rows.append({
            "Algorithm": model_name,
            "Feature set": set_name,
            "Balanced accuracy": balanced_accuracy_score(y_test, pred),
            "F1 (0)": f1_0,
            "F1 (1)": f1_1,
        })

pd.DataFrame(rows).round(2)

Train: {1: 682, 0: 225} | Test: {1: 171, 0: 56}


,Algorithm,Feature set,Balanced accuracy,F1 (0),F1 (1)
0,Logistic Regression,OOV + PPL,0.57,0.39,0.73
1,Logistic Regression,OOV Gain + PPL Gain,0.69,0.53,0.81
2,Logistic Regression,All four features,0.70,0.54,0.81
3,Random Forest,OOV + PPL,0.59,0.39,0.77
4,Random Forest,OOV Gain + PPL Gain,0.68,0.51,0.78
5,Random Forest,All four features,0.70,0.54,0.78


## Final model (Table 4)

Logistic regression with all four features.

In [6]:
final_features = FEATURE_SETS["All four features"]
scaler, model, pred = fit_and_predict("Logistic Regression", final_features)
y_test = y[idx_test]

print(confusion_matrix(y_test, pred, labels=[0, 1]))
print(classification_report(y_test, pred, digits=2))

for name, coef in zip(final_features, model.coef_[0]):
    print(f"{name}: {coef:.3f}")

[[ 36  20]
 [ 41 130]]
              precision    recall  f1-score   support

           0       0.47      0.64      0.54        56
           1       0.87      0.76      0.81       171

    accuracy                           0.73       227
   macro avg       0.67      0.70      0.68       227
weighted avg       0.77      0.73      0.74       227

Gain_Score_OOV: -0.281
Gain_Score_Perplexite_log: -0.859
Score_OOV_apres: -0.331
Perplexite_log: -0.458


### Error analysis

False negatives for class 0: noisy examples (true 0) predicted as analyzable (1).

In [7]:
results = df.loc[idx_test, ["doc_id", "PivotSent", "pivot_sent_corrected",
                             "annot_preCorrection", TARGET] + final_features].copy()
results["prediction"] = pred

false_negatives = results[(results[TARGET] == 0) & (results["prediction"] == 1)]
print(len(false_negatives), "false negatives for class 0")
false_negatives.to_csv(OUTPUT_DIR / "false_negatives_class_0.csv", index=False)

20 false negatives for class 0


## Save the final model

In [8]:
joblib.dump(
    {
        "scaler": scaler,
        "model": model,
        "feature_cols": final_features,
        "sklearn_version": sklearn.__version__,
    },
    OUTPUT_DIR / "best_model.pkl",
)

['../outputs/best_model.pkl']